# SATD Classification on New Issue Data

Runs trained Pipeline A models on new issue data with 3 variations:
1. Title only
2. Description only  
3. Title + Description

**Models used:**
- Embedding: Qwen3-Embedding-0.6B (frozen encoder)
- Classifier: XGBoost (best performer)

**Output:** Parquet file with `ID, Title, Description, Title_SATD, Description_SATD, Title_Description_SATD`

## Kaggle Setup
1. Upload `issue_202608272234.parquet` as a dataset
2. Upload `models/` folder as a dataset
3. Add both datasets to this notebook

In [14]:
!pip install -q pandas pyarrow sentence-transformers xgboost gdown torch

In [16]:
import os
import pandas as pd
import numpy as np
import joblib
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## 1. Configuration

In [17]:
# === KAGGLE PATHS ===
# Update these paths based on how you added the datasets
INPUT_PATH = "/kaggle/input/datasets/musaddiqrafi/issue-paraquet/issue_202608272234.parquet"  # UPDATE THIS
MODEL_DIR = "/kaggle/input/datasets/musaddiqrafi/models/models"  # UPDATE THIS
OUTPUT_DIR = "/kaggle/working"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "satd_classification_results.parquet")

# Model paths
LOCAL_IDENTIFICATION_MODEL = os.path.join(MODEL_DIR, "identification_issue_qwen3_xgboost.joblib")
LOCAL_CATEGORIZATION_MODEL = os.path.join(MODEL_DIR, "categorization_issue_qwen3_xgboost.joblib")

print(f"Input: {INPUT_PATH}")
print(f"Models: {MODEL_DIR}")
print(f"Output: {OUTPUT_PATH}")

Input: /kaggle/input/datasets/musaddiqrafi/issue-paraquet/issue_202608272234.parquet
Models: /kaggle/input/datasets/musaddiqrafi/models/models
Output: /kaggle/working/satd_classification_results.parquet


## 2. Load Data

In [18]:
df = pd.read_parquet(INPUT_PATH)
print(f"Loaded {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")
df.head()

Loaded 458232 rows
Columns: ['ID', 'Title', 'Description']


,ID,Title,Description
0,912,Create a Java client for Receptor,"As a developer, I'd like to create a [java cli..."
1,913,Create Boot based ModuleRunner,"As a developer, I'd like to build isolated Boo..."
2,914,Create a pluggable runtime SPI,"As a developer, I'd like to migrate module dep..."
3,915,Simplify GPDB UX around parameters for the Sqo...,"As a developer, I'd like to have a simplified ..."
4,916,Sqoop Module not running,Can not get any jobs created using Scoop Modu...


In [19]:
print("Missing values:")
print(df.isnull().sum())

df['Title'] = df['Title'].fillna('')
df['Description'] = df['Description'].fillna('')

Missing values:
ID                 0
Title              0
Description    29128
dtype: int64


## 3. Load Embedding Model

In [25]:
import torch
from sentence_transformers import SentenceTransformer

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

print("Loading Qwen3-Embedding-0.6B...")

embedder = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    device="cuda"
)

print("Model loaded!")

CUDA: True
GPU: Tesla T4
Loading Qwen3-Embedding-0.6B...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Model loaded!


In [26]:
!curl -I https://huggingface.co

HTTP/2 200 
content-type: text/html; charset=utf-8
content-length: 177828
date: Mon, 07 Sep 2026 20:31:31 GMT
etag: W/"2b6a4-gWa3qWzFIJWdsBKE1AmcJPYKeW4"
x-powered-by: huggingface-moon
x-request-id: Root=1-6a9f1f23-0bcd9412054a12204854f8ec
ratelimit: "pages";r=99;t=184
ratelimit-policy: "fixed window";"pages";q=100;w=300
cross-origin-opener-policy: same-origin
referrer-policy: strict-origin-when-cross-origin
link: </.well-known/ai-catalog.json>; rel="ai-catalog"; type="application/ai-catalog+json"
link: </.well-known/api-catalog>; rel="api-catalog"
x-frame-options: DENY
x-cache: Hit from cloudfront
via: 1.1 30c6d7b3beb0a918037485bea6e0158c.cloudfront.net (CloudFront)
x-amz-cf-pop: ORD58-P13
alt-svc: h3=":443"; ma=86400
x-amz-cf-id: -oaGyUPq8dSWzRU09ZSnDYkIlqGryj_9g2ZiVWnBpR-TpUPv_F5QHw==
age: 44
strict-transport-security: max-age=31536000



## 4. Generate Embeddings for 3 Variations

In [ ]:
def get_embeddings(texts, batch_size=128):
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch = texts[i:i+batch_size]
        emb = embedder.encode(batch, show_progress_bar=False, normalize_embeddings=True, batch_size=128)
        embeddings.append(emb)
    return np.vstack(embeddings)

print("Generating embeddings for 3 variations...")

print("\n1. Title only...")
title_embeddings = get_embeddings(df['Title'].tolist(), batch_size=128)
print(f"   Shape: {title_embeddings.shape}")

print("\n2. Description only...")
desc_embeddings = get_embeddings(df['Description'].tolist(), batch_size=128)
print(f"   Shape: {desc_embeddings.shape}")

print("\n3. Title + Description...")
title_desc_text = (df['Title'] + " " + df['Description']).tolist()
title_desc_embeddings = get_embeddings(title_desc_text, batch_size=128)
print(f"   Shape: {title_desc_embeddings.shape}")

print("\nAll embeddings generated!")

Generating embeddings for 3 variations...

1. Title only...


Embedding:   3%|▎         | 102/3580 [02:55<1:39:02,  1.71s/it]

## 5. Load Trained Classifiers

In [ ]:
print("Loading classification models...")
identification_model = joblib.load(LOCAL_IDENTIFICATION_MODEL)
categorization_model = joblib.load(LOCAL_CATEGORIZATION_MODEL)
print("Models loaded!")

print(f"\nIdentification model classes: {identification_model.classes_}")
print(f"Categorization model classes: {categorization_model.classes_}")

## 6. Run Classification

Two-step process:
1. **Identification**: Is this SATD or Not-SATD? (binary)
2. **Categorization**: If SATD, what type? (C/D, REQ, TES, DOC)

In [ ]:
def classify_embeddings(embeddings, identification_model, categorization_model):
    """
    Run two-step classification:
    1. Identify SATD vs Not-SATD
    2. Categorize SATD items
    
    Returns: list of strings (e.g., 'Not-SATD', 'C/D', 'DOC', etc.)
    """
    # Step 1: Identification
    identification_preds = identification_model.predict(embeddings)
    
    # Step 2: Categorization for SATD items
    results = identification_preds.copy()
    satd_mask = identification_preds == 'SATD'
    
    if satd_mask.sum() > 0:
        categorization_preds = categorization_model.predict(embeddings[satd_mask])
        results[satd_mask] = categorization_preds
    
    return results.tolist()

print("Running classification on 3 variations...")

print("\n1. Title only...")
title_satd = classify_embeddings(title_embeddings, identification_model, categorization_model)
print(f"   Done. SATD count: {sum(1 for x in title_satd if x != 'Not-SATD')}")

print("\n2. Description only...")
desc_satd = classify_embeddings(desc_embeddings, identification_model, categorization_model)
print(f"   Done. SATD count: {sum(1 for x in desc_satd if x != 'Not-SATD')}")

print("\n3. Title + Description...")
title_desc_satd = classify_embeddings(title_desc_embeddings, identification_model, categorization_model)
print(f"   Done. SATD count: {sum(1 for x in title_desc_satd if x != 'Not-SATD')}")

print("\nAll classifications complete!")

## 7. Save Results

In [ ]:
df['Title_SATD'] = title_satd
df['Description_SATD'] = desc_satd
df['Title_Description_SATD'] = title_desc_satd

df.to_parquet(OUTPUT_PATH, index=False)
print(f"Results saved to: {OUTPUT_PATH}")
print(f"Total rows: {len(df)}")

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
for col in ['Title_SATD', 'Description_SATD', 'Title_Description_SATD']:
    print(f"\n{col}:")
    print(df[col].value_counts())

In [ ]:
df.head(10)